# Clustering and PCA — Solution

**Dataset:** Car models described by engine size, fuel consumption and CO₂ emissions; no target is used.

Run from top to bottom. The steps match the exercise; short answers explain the results.

## Libraries and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

## 1. Move a cluster centre

For points (1,1) and (3,1), replace the gap with `points.mean(axis=0)`. What is the new centre and WCSS?

In [ ]:
points = np.array([[1, 1], [3, 1]])
centre = points.mean(axis=0)
print('New centre:', centre)
print('WCSS:', np.sum((points - centre) ** 2))

**Answer:** The centre is (2,1); each point is distance 1 away, so WCSS = 2.

## 2. Prepare and scale car features

Preparation is provided. Are any of these variables a target? Why scale before grouping?

In [ ]:
cars = pd.read_csv('../00_Data/car_fuel_consumption_2025.csv')
features = ['engine_size', 'combined', 'co2']
cars = cars.dropna(subset=features).copy()
X = cars[features]
scaler = StandardScaler()
scaled = scaler.fit_transform(X)
print(X.head())
print('Scaled rows:')
print(np.round(scaled[:5], 2))

**Answer:** All three variables are inputs. There is no target. They have different units and spreads; scaling prevents a numerically large variable from dominating distances. For this descriptive task, we group the available rows rather than estimating supervised test performance.

## 3. Fit three clusters

Replace the gap with `KMeans(n_clusters=3, n_init=10, random_state=42)`. Look at the plot: are groups perfectly separated?

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
labels = kmeans.fit_predict(scaled)
cars['cluster'] = labels
plt.scatter(cars['engine_size'], cars['combined'], c=labels, cmap='viridis', alpha=0.5)
plt.xlabel('Engine size [L]')
plt.ylabel('Fuel consumption [L/100 km]')
plt.title('Car models, coloured by cluster')
plt.show()

**Answer:** The clusters represent similar scaled feature combinations and can overlap in a two-input projection. Three clusters are an illustrative starting choice, not a known true classification.

## 4. Read an elbow plot

The short loop is provided. Where do improvements become smaller? Why is the smallest WCSS alone not a good rule for choosing k?

In [ ]:
ks = range(1, 7)
wcss = []
for k in ks:
    candidate = KMeans(n_clusters=k, n_init=10, random_state=42)
    candidate.fit(scaled)
    wcss.append(candidate.inertia_)
plt.plot(list(ks), wcss, marker='o')
plt.xlabel('Number of clusters k')
plt.ylabel('Within-cluster sum of squares')
plt.show()

**Answer:** An elbow can guide a small useful k, but may be unclear. More clusters reduce within-cluster variation; choosing only the smallest WCSS would keep adding clusters. Interpretation also matters.

## 5. Describe the clusters

The table is supplied. Give each cluster a description based on its average engine size, consumption and CO₂. Does cluster 0 mean “best”?

In [ ]:
print('Cluster sizes:')
print(cars['cluster'].value_counts().sort_index())
print('Mean values in original units:')
print(cars.groupby('cluster')[features].mean().round(2))
print('Cluster centres in original units:')
print(pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features).round(2))

**Answer:** Describe groups using the printed means, such as smaller/lower-consumption or larger/higher-consumption cars. Labels 0,1,2 are arbitrary identifiers, not quality rankings. The inverse-scaled centres match the cluster means.

## 6. Read a dendrogram

For a readable demonstration we use 15 supplied car rows. Which observations merge early? What does a high merge mean? The cluster cut is provided.

In [ ]:
sample = cars.sample(15, random_state=42)
sample_scaled = scaler.transform(sample[features])
Z = linkage(sample_scaled, method='ward')
names = (sample['make'] + ' ' + sample['model']).tolist()
plt.figure(figsize=(11, 5))
dendrogram(Z, labels=names, leaf_rotation=90)
plt.ylabel('Ward linkage height')
plt.tight_layout()
plt.show()
print('Three-cluster cut:', fcluster(Z, t=3, criterion='maxclust'))

**Answer:** Early merges join similar rows under Ward linkage. A high merge joins more separated groups. A horizontal cut groups the branches below it; this demonstration describes only the sampled rows.

## 7. PCA outlook

The PCA code is supplied. How much variance do PC1 and PC2 retain together? Why is this percentage different from prediction accuracy?

In [ ]:
pca = PCA(n_components=2)
components = pca.fit_transform(scaled)
print('Explained variance ratios:', pca.explained_variance_ratio_.round(3))
print('Total retained:', round(pca.explained_variance_ratio_.sum(), 3))
print('Loadings:')
print(pd.DataFrame(pca.components_, columns=features, index=['PC1', 'PC2']).round(3))
plt.scatter(components[:, 0], components[:, 1], c=labels, cmap='viridis', alpha=0.5)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA projection; colours from the earlier k-means')
plt.show()

**Answer:** Add the two printed variance ratios. They measure how much input spread remains after projecting to two axes, not how many predictions are correct. PCA did not use the cluster labels; the colours are added afterwards. Component signs may be flipped without changing the information.

### Jupyter notebook — footer info

Include this information when submitting your notebook.

In [ ]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))